# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

https://colab.research.google.com/github/jasontpv/DS2002FA26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


In [2]:
# File not found, so I had to write own file path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
path_to_DS_2002_folder = '/content/drive/MyDrive/DS-2002'

os.chdir(path_to_DS_2002_folder)

cat = pd.read_csv('./data/product_catalog.csv')
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


In [4]:
cat.shape

(45, 7)

### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [5]:
pop_tart_products = cat.loc[cat['product_name'].str.lower().str.contains('pop', na=False)]

# Print the count
print(f'There are {len(pop_tart_products)} rows that look like Pop-Tart products.')

# Print the matching rows
pop_tart_products

There are 4 rows that look like Pop-Tart products.


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [6]:
# Strawberry
strawberry = pop_tart_products.loc[pop_tart_products['product_name'].str.lower().str.contains('strawberry', na=False)]

# List SKUs and names
print('SKUs and names that all mean strawberry Pop-Tart\n', strawberry[['sku', 'product_name']], '\n')

# Not strawberry
not_strawberry = pop_tart_products.loc[~pop_tart_products['product_name'].str.lower().str.contains('strawberry', na=False)]

print('SKUs and names that all mean non-strawberry Pop-Tart\n', not_strawberry[['sku', 'product_name']])

SKUs and names that all mean strawberry Pop-Tart
              sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry 

SKUs and names that all mean non-strawberry Pop-Tart
 Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** None of the 4 Pop-Tart rows appear to be a different product. All four rows have names that suggests they are strawberry Pop-Tarts.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [7]:
# SKU mapping to one canonical SKU (PT-12)
sku_mapping = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12'
}

# Apply it in a way that leaves unmapped SKUs as themselves rather than NaN values
cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,PT-12
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,PT-12
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24


In [8]:
# Confirm result by verifying the number of unique original and canonical SKUs
print("Original unique SKUs:", cat['sku'].nunique())
print("Canonical unique SKUs:", cat['canonical_sku'].nunique())

# Changed 3 SKUs to map to PT-12 so correct

Original unique SKUs: 45
Canonical unique SKUs: 42


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [9]:
print('Distinct values before normalization\n', cat['category'].value_counts(dropna=False))

Distinct values before normalization
 category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64


In [10]:
## Normalize category column
# Strip and title-case
cat['category'] = cat['category'].str.strip().str.title()

In [11]:
print('Distinct values after normalization\n', cat['category'].value_counts(dropna=False))

Distinct values after normalization
 category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1
Alcohol                1
Alcoholic Beverages    1
Groceries              1
Baby Products          1
Name: count, dtype: int64


In [12]:
# Continuation of normalization

# Identify labels that represent the same category
# Breakfast and Breakfast Foods
# Alcohol and Alcoholic Beverages
# Grocery and Groceries
# Emergency and Emergency Supplies
# Baby and Baby Products

cat['category'] = cat['category'].replace({
    'Breakfast Foods': 'Breakfast',
    'Alcoholic Beverages': 'Alcohol',
    'Groceries': 'Grocery',
    'Emergency Supplies': 'Emergency',
    'Baby Products': 'Baby'
})

print('Distinct values after normalization\n', cat['category'].value_counts(dropna=False))

Distinct values after normalization
 category
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [13]:
## Build a candidate list programmatically

# Normalize each product name (lowercase, strip punctuation, sort the words)
cat['normalized'] = (
    cat['product_name'].str.lower()
    .str.replace(r'[^a-z ]', '', regex=True)
    .str.split()
    .apply(lambda w: ' '.join(sorted(w)))
)

# Group by normalized name
candidates = (
    cat.groupby('normalized')['sku']
    .agg(list)
    .loc[lambda s: s.str.len() > 1]
)

print('Groups holding more than one SKU:')
candidates

Groups holding more than one SKU:


,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [14]:
# Assert that appplying mapping produced no nulls in the canonical column
assert cat['canonical_sku'].notnull().all()

# Assert that applying mapping did not change the number of rows
original_row_count = len(cat)
assert len(cat) == original_row_count

print('Mapping produced no nulls in the canonical column and did not change the number of rows.')

Mapping produced no nulls in the canonical column and did not change the number of rows.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

In [15]:
water_products = cat.loc[cat['product_name'].str.lower().str.contains('water', na=False)]

# Print the count
print(f'There are {len(water_products)} rows that look like water-related products.')

# Print the matching rows
water_products

There are 3 rows that look like water-related products.


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku,normalized
4,WTR-24,BOTTLED WATER 24PK,Emergency,Bottled,2.64,Sara Lee,1 pack,WTR-24,bottled pk water
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,WATER24PK,bottled pack water
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,WTR_24PK,pack water


In [16]:
baby_products = cat.loc[cat['product_name'].str.lower().str.contains('baby', na=False)]

# Print the count
print(f'There are {len(baby_products)} rows that look like baby-related products.')

# Print the matching rows
baby_products

There are 2 rows that look like baby-related products.


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku,normalized
39,DIAP-S3,Baby Diapers - Size 3,Baby,Diapers,14.40,Procter & Gamble,single,DIAP-S3,baby diapers size
42,BABYFORMULA,Baby Formula,Baby,NaN,18.25,Sara Lee,6 pack,BABYFORMULA,baby formula


The water product family appears fragmented. After filtering for product names containing 'water', there are 3 rows, including 'BOTTLED WATER 24PK', 'Bottled Water 24-Pack', and 'Water - 24 Pack'. Because these product names appear to represent the same product, I would group them under one canonical SKU after confirming that they are all 24-packs of water.

The baby product family also contains multiple products. After filtering for product names containing 'baby', there are 2 rows, including 'Baby Diapers - Size 3' and 'Baby Formula'. These products could be grouped under a broader Baby category. However, they should continue to have different canonical SKU values becuase they are different products.



Consolidation stops becoming a data-cleaning task and becomes a business decision when the available data/information does not provide enough information to determine whether different entries should be treated as the same product. For example, the baby product family contains 2 products, namely 'Baby Diapers - Size 3' and 'Baby Formula'. These products could be grouped under a broader Baby category, but they should continue to have different canonical SKU values becuase they are different products. This is a decision that needs to be made with more insight as to what this data will be used for in the future. To clarify, I would ask the inventory team/manager.